# Lab 5 &middot; Finding corners

BITS F459 Computer Vision. **8 marks.** Part A 5, Part B 3.

Last week you found edges. An edge tells you there is a boundary, but it does not tell you
*where along it* you are: slide a window along an edge and nothing changes. A corner is a
point where the window cannot slide in any direction without the view changing, which is
what makes it a place you can find again in another photograph.

This lab does two things.

**Part A** works Harris by hand on a small board your own BITS ID generates. Every number
is a whole number, so there is no rounding anywhere.

**Part B** asks what survives when the picture changes. You will watch a threshold that
works perfectly on one picture find nothing at all on the same picture at lower contrast.

You may work with other people. The board is generated from your BITS ID, so the numbers
are yours alone.

In [ ]:
BITS_ID = ""      # your BITS ID, exactly as on your ID card, e.g. "2024A7PS0123U"
assert BITS_ID.strip(), "Put your BITS ID in the line above before running anything else."

In [ ]:
import hashlib, json
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

SEED = hashlib.sha256(BITS_ID.strip().upper().encode()).hexdigest()
np.set_printoptions(linewidth=200, suppress=True)

SX = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
SY = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float)
K  = 0.04

def gradients(img):
    # correlate, not convolve: the kernel is applied exactly as it is written on
    # the slide, which is what you will do by hand. convolve() flips it and
    # returns the opposite sign.
    gx = ndimage.correlate(img, SX, mode="nearest") / 8.0
    gy = ndimage.correlate(img, SY, mode="nearest") / 8.0
    return gx, gy

def window_sums(gx, gy):
    box = lambda a: ndimage.uniform_filter(a, 3, mode="constant") * 9.0
    return box(gx*gx), box(gy*gy), box(gx*gy)

def response(A, B, C, k=K):
    return (A*B - C*C) - k*(A + B)**2

def harris(img, k=K):
    gx, gy = gradients(img)
    A, B, C = window_sums(gx, gy)
    return response(A, B, C, k)

def peaks(R, n=None, thresh=None, radius=1):
    # non-maximum suppression: a pixel survives only if nothing around it beats it
    mx = ndimage.maximum_filter(R, size=2*radius+1, mode="constant")
    keep = (R == mx) & (R > 0)
    if thresh is not None:
        keep &= (R >= thresh)
    rc = np.argwhere(keep)
    if n is not None and len(rc) > n:
        rc = rc[np.argsort([-R[r, c] for r, c in rc])][:n]
    return [tuple(map(int, x)) for x in rc]

print("ready")

In [ ]:
# ---- your board, generated from your BITS ID -------------------------------
BLOCK, GRID = 3, 4
LOWS, STEPS = (24, 32, 40, 48), (96, 112, 128, 144, 160)

def _alt(pat):
    out = []
    for i in range(pat.shape[0]-1):
        for j in range(pat.shape[1]-1):
            q = pat[i:i+2, j:j+2]
            if q[0,0] == q[1,1] and q[0,1] == q[1,0] and q[0,0] != q[0,1]:
                out.append((i, j))
    return out

_r = np.random.default_rng(int(SEED[8:16], 16))
LO = int(_r.choice(LOWS)); HI = LO + int(_r.choice(STEPS))
for _ in range(4000):
    _pat = _r.integers(0, 2, (GRID, GRID))
    if not _alt(_pat):
        continue
    BOARD = np.where(np.kron(_pat, np.ones((BLOCK, BLOCK))), HI, LO).astype(float)
    _gx, _gy = gradients(BOARD)
    _A, _B, _C = window_sums(_gx, _gy)
    _R = response(_A, _B, _C)
    _cor = [tuple(int(v) for v in x) for x in np.argwhere(
        (_A == _B) & (_A > 0) & (_C != 0) & (_R > 0) & (_gx != 0) & (_gy != 0))]
    _edg = [tuple(int(v) for v in x) for x in np.argwhere((((_A > 0) & (_B == 0)) | ((_A == 0) & (_B > 0))) & (_C == 0) & (_R < 0))]
    _flt = [tuple(int(v) for v in x) for x in np.argwhere((_A == 0) & (_B == 0) & (_C == 0))]
    _surv = set(peaks(_R))
    _cor = [p for p in _cor if p in _surv
            and 1 <= p[0] < BOARD.shape[0]-1 and 1 <= p[1] < BOARD.shape[1]-1]
    if _cor and _edg and _flt:
        CORNER = max(_cor, key=lambda p: _R[p])
        EDGE = _edg[int(_r.integers(len(_edg)))]
        FLAT = _flt[int(_r.integers(len(_flt)))]
        break

print(f"your board is {BOARD.shape[0]} by {BOARD.shape[1]}, light {HI}, dark {LO}\n")
print(BOARD.astype(int))
print(f"\n  the corner  you work:  row {CORNER[0]}, column {CORNER[1]}")
print(f"  the edge    you work:  row {EDGE[0]}, column {EDGE[1]}")
print(f"  the flat    you work:  row {FLAT[0]}, column {FLAT[1]}")

plt.figure(figsize=(3.2, 3.2))
plt.imshow(BOARD, cmap="gray", vmin=0, vmax=255)
for p, m, col in ((CORNER, "o", "red"), (EDGE, "s", "orange"), (FLAT, "^", "cyan")):
    plt.plot(p[1], p[0], m, ms=9, mfc="none", mew=2, color=col)
plt.title("your board: corner (o), edge (s), flat (^)", fontsize=9)
plt.axis("off"); plt.show()

---

# Part A &middot; Harris by hand, on your own board &nbsp;&nbsp;<small>5 marks</small>

Everything in Part A is done with pen and paper from the board printed above, then typed
in. Nothing here needs the computer to find the answer for you.

The five steps, in the order the lecture gave them:

1. **Differentiate.** Sobel divided by 8, applied to the board itself. There is no Gaussian
   smoothing first. Canny smooths and then differentiates; Harris differentiates and then
   smooths. That order is the whole difference.
2. **Multiply.** Form `Ix²`, `Iy²` and `IxIy` at every pixel.
3. **Sum over a window.** Add each of those three over the 3 by 3 window around the pixel.
   Those three sums are the matrix `M`.
4. **Score.** `R = det(M) − k·trace(M)²`, with `k = 0.04`.
5. **Thin.** Non-maximum suppression, so one corner gives one answer rather than a blob.

At the edge of the board, the row or column outside is a copy of the nearest one inside.
This is the same replicate rule you used in Lab 4.

## A1 &middot; the two gradients &nbsp;&nbsp;<small>1 mark</small>

Work out `Ix` and `Iy` at your **corner** pixel, by hand.

`Ix` is the Sobel `SX` kernel laid over the 3 by 3 window of the board centred on that
pixel, multiplied entry by entry, added up, and divided by 8. `Iy` is the same with `SY`.

Both answers are whole numbers.

In [ ]:
A1 = {
    "Ix": None,     #
    "Iy": None,     #
}
assert all(v is not None for v in A1.values()), "Fill in both."
print(f"  at the corner ({CORNER[0]},{CORNER[1]}):  Ix = {A1['Ix']}   Iy = {A1['Iy']}")

## A2 &middot; the matrix M, at three places &nbsp;&nbsp;<small>2 marks</small>

Build `M` at all three of your marked pixels: the corner, the edge and the flat one.

`M` has four entries but only three different numbers, because it is symmetric:

```
        M  =  [  sum of Ix²      sum of IxIy  ]
              [  sum of IxIy     sum of Iy²   ]
```

Each sum runs over the 3 by 3 window centred on the pixel. So for each of the three places
you need nine values of `Ix` and nine of `Iy`, then three sums.

Give `Ixx`, `Iyy` and `Ixy` for each. All are whole numbers.

Both marks need all three places right. One mark for two of the three.

In [ ]:
A2 = {
    "corner": {"Ixx": None, "Iyy": None, "Ixy": None},    #
    "edge":   {"Ixx": None, "Iyy": None, "Ixy": None},    #
    "flat":   {"Ixx": None, "Iyy": None, "Ixy": None},    #
}
for _k, _p in (("corner", CORNER), ("edge", EDGE), ("flat", FLAT)):
    _e = A2[_k]
    assert all(v is not None for v in _e.values()), f"Fill in all three for the {_k}."
    print(f"  {_k:7s} ({_p[0]},{_p[1]})   M = [[{_e['Ixx']:.0f}, {_e['Ixy']:.0f}], "
          f"[{_e['Ixy']:.0f}, {_e['Iyy']:.0f}]]")

## A3 &middot; the eigenvalues, and R &nbsp;&nbsp;<small>1 mark</small>

At your corner the matrix has the form

```
        M  =  [  a   b  ]
              [  b   a  ]
```

with the **same number on both diagonal entries**. For that shape the eigenvalues are
simply `a − b` and `a + b`. No quadratic formula is needed. Give the smaller one as
`lam2` and the larger as `lam1`.

Then the Harris response, with `k = 0.04`:

```
        R  =  det(M)  −  k · trace(M)²
           =  (a² − b²)  −  0.04 · (2a)²
```

`R` is not a whole number in general. Give it **rounded to the nearest whole number**.

In [ ]:
A3 = {
    "lam2": None,     #  the smaller eigenvalue
    "lam1": None,     #  the larger eigenvalue
    "R":    None,     #  rounded to the nearest whole number
}
assert all(v is not None for v in A3.values()), "Fill in all three."
print(f"  lam2 {A3['lam2']:.0f}   lam1 {A3['lam1']:.0f}   R {A3['R']:,}")

## A4 &middot; the three verdicts, and thinning &nbsp;&nbsp;<small>1 mark</small>

Classify each of your three pixels from the sign of `R` alone:

| `R` | verdict |
|:--|:--|
| clearly positive | `"corner"` |
| clearly negative | `"edge"` |
| zero | `"flat"` |

You already have `R` at the corner. For the edge and the flat pixel you can read the
verdict straight off the shape of `M`, without computing `R` in full: think about what
`det(M)` is when one of the diagonal entries is zero.

Then step 5. Run the cell below to see the whole `R` map with non-maximum suppression
applied. Report how many pixels survive, and whether your corner is one of them.

In [ ]:
_Rmap = harris(BOARD)
_surv = peaks(_Rmap)
print("R over the whole board, in millions:\n")
print(np.round(_Rmap/1e6, 1))
print(f"\n  {len(_surv)} pixels survive non-maximum suppression: {_surv}")
print(f"  your corner ({CORNER[0]},{CORNER[1]}) is "
      f"{'among them' if CORNER in _surv else 'NOT among them'}")

In [ ]:
A4 = {
    "corner_verdict": None,    #  "corner", "edge" or "flat"
    "edge_verdict":   None,    #
    "flat_verdict":   None,    #
    "n_survivors":    None,    #  how many pixels survive step 5
    "why_thinning":   None,    #  one sentence: what step 5 is for
}
assert all(v is not None for v in A4.values()), "Fill in all five."
assert len(str(A4["why_thinning"]).strip()) > 40, "Write a proper sentence."
print("  ", A4)

---

# Part B &middot; what survives a change of picture &nbsp;&nbsp;<small>3 marks</small>

Harris gives every pixel a score. To turn scores into a list of corners you have to decide
which scores count. The obvious way is to pick a number and keep everything above it.

This part asks whether that number travels.

Below, the same board is shown three ways: as it is, turned by 90 degrees, and at **half
the contrast** (the light and dark values moved closer together, so the picture looks
washed out but every shape is in exactly the same place).

In [ ]:
ROT  = np.rot90(BOARD)
HALF = (BOARD - BOARD.min())/2.0 + BOARD.min()

fig, ax = plt.subplots(1, 3, figsize=(9, 3.1))
for a, im, t in ((ax[0], BOARD, "as it is"), (ax[1], ROT, "turned 90 degrees"),
                 (ax[2], HALF, "half the contrast")):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

R_BOARD, R_ROT, R_HALF = harris(BOARD), harris(ROT), harris(HALF)
TRUE_CORNERS = peaks(R_BOARD)
print(f"  on the original board, {len(TRUE_CORNERS)} corners survive thinning")
print(f"  the weakest of them scores {min(R_BOARD[p] for p in TRUE_CORNERS):,.0f}")
print(f"  the same corner on the half-contrast picture scores "
      f"{min(R_HALF[p] for p in TRUE_CORNERS):,.0f}")

## B1 &middot; what the contrast did to R &nbsp;&nbsp;<small>1 mark</small>

Give `R` at your own corner on the original board and on the half-contrast one, each
rounded to the nearest whole number, and the ratio of the first to the second.

The ratio is not 2. Say in one sentence why it is what it is, in terms of how many
gradients are multiplied together to make `R`.

In [ ]:
B1 = {
    "R_full": None,     #
    "R_half": None,     #
    "ratio":  None,     #  R_full divided by R_half, to the nearest whole number
    "because": None,    #  one sentence
}
assert all(v is not None for v in B1.values()), "Fill in all four."
assert len(str(B1["because"]).strip()) > 40, "Write a proper sentence."
print("  ", {k: v for k, v in B1.items() if k != "because"})

## B2 &middot; a threshold that does not travel &nbsp;&nbsp;<small>1 mark</small>

Choose a threshold `T` that, applied to the **original** board, keeps every one of its
corners and nothing else. Then apply that same `T`, unchanged, to the turned picture and
to the half-contrast picture, and report how many corners each keeps.

The cell checks your `T` against the original for you. The other two counts are the point
of the question.

In [ ]:
B2 = {
    "T": None,          #  a threshold on R
}
assert B2["T"] is not None, "Choose a threshold."
_n_full = len(peaks(R_BOARD, thresh=B2["T"]))
_n_rot  = len(peaks(R_ROT,   thresh=B2["T"]))
_n_half = len(peaks(R_HALF,  thresh=B2["T"]))
print(f"  T = {B2['T']:,}")
print(f"    original         {_n_full:3d} corners   (there are {len(TRUE_CORNERS)})")
print(f"    turned 90        {_n_rot:3d} corners")
print(f"    half contrast    {_n_half:3d} corners")
print(f"  {'T keeps exactly the right corners on the original' if _n_full == len(TRUE_CORNERS) else 'T does NOT keep the right corners on the original - adjust it'}")

## B3 &middot; the fix &nbsp;&nbsp;<small>1 mark</small>

Instead of a threshold, keep the **strongest N** responses, where `N` is the number of
corners you expect. Run it on all three pictures and report how many of the original's
corners come back each time.

Then say, in one sentence, which of the two rules you would use on photographs taken in
different light, and why.

In [ ]:
_N = len(TRUE_CORNERS)
_top_full = set(peaks(R_BOARD, n=_N))
_top_half = set(peaks(R_HALF,  n=_N))
print(f"  keeping the strongest {_N}:")
print(f"    original        {len(_top_full & set(TRUE_CORNERS))} of {_N} recovered")
print(f"    half contrast   {len(_top_half & set(TRUE_CORNERS))} of {_N} recovered")

B3 = {
    "recovered_full": None,    #
    "recovered_half": None,    #
    "which_rule":     None,    #  one sentence
}
assert all(v is not None for v in B3.values()), "Fill in all three."
assert len(str(B3["which_rule"]).strip()) > 40, "Write a proper sentence."
print("  ", {k: v for k, v in B3.items() if k != "which_rule"})

---

## Packing your answers

Run the cell below **last**, after every other cell has been run. It gathers everything
into one block and recomputes Part B from your own settings.

If you go back and change anything, run this cell again afterwards. The block is what gets
marked, not the printed output above it.

In [ ]:
ANSWERS = {
    "lab": "lab05", "version": 1,
    "bits_id": BITS_ID.strip().upper(), "seed": SEED[:8],
    "generated": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "board": {"hi": int(HI), "lo": int(LO),
              "corner": list(CORNER), "edge": list(EDGE), "flat": list(FLAT)},
    "A1": {k: float(v) for k, v in A1.items()},
    "A2": {p: {k: float(v) for k, v in d.items()} for p, d in A2.items()},
    "A3": {"lam2": float(A3["lam2"]), "lam1": float(A3["lam1"]), "R": int(A3["R"])},
    "A4": {"corner_verdict": str(A4["corner_verdict"]).strip().lower(),
           "edge_verdict":   str(A4["edge_verdict"]).strip().lower(),
           "flat_verdict":   str(A4["flat_verdict"]).strip().lower(),
           "n_survivors": int(A4["n_survivors"]), "why_thinning": str(A4["why_thinning"])},
    "B1": {"R_full": int(B1["R_full"]), "R_half": int(B1["R_half"]),
           "ratio": int(B1["ratio"]), "because": str(B1["because"])},
    "B2": {"T": int(B2["T"]),
           "n_full": int(len(peaks(R_BOARD, thresh=int(B2["T"])))),
           "n_rot":  int(len(peaks(R_ROT,   thresh=int(B2["T"])))),
           "n_half": int(len(peaks(R_HALF,  thresh=int(B2["T"]))))},
    "B3": {"recovered_full": int(B3["recovered_full"]),
           "recovered_half": int(B3["recovered_half"]),
           "which_rule": str(B3["which_rule"])},
}

print("===== LAB05 ANSWER BLOCK v1 =====")
print(json.dumps(ANSWERS, separators=(",", ":"), default=float))
print("===== END LAB05 ANSWER BLOCK =====")
print("\nPacked. Now download this notebook as lab05.ipynb and upload it to your repo.")

### Handing it in

1. **Run every cell from top to bottom, and run the packing cell last.**
2. **File &rarr; Download &rarr; Download .ipynb**
3. **Rename the file to exactly `lab05.ipynb`.**
4. Open your repository: `github.com/BITS-F459-Computer-Vision/f459-<your BITS ID, lowercase>`
5. **Add file &rarr; Upload files**, drag it in, **Commit changes**.
6. **Click the file on GitHub and check you can see your outputs in it.**

> The packed block is what gets marked. If you change an answer and do not run the packing
> cell again, the block still holds the old one. This was set out in the Lab 3 feedback and
> it is not accepted from Lab 4 onwards.

### Before you go

- [ ] A1, A2, A3, A4 all filled in and printing without an error
- [ ] B2's threshold keeps the right number of corners on the original
- [ ] B1 and B3 filled in, with the written sentences actually written
- [ ] Packing cell run **last**, its output visible
- [ ] Downloaded, renamed to exactly `lab05.ipynb`, uploaded
- [ ] You clicked the file on GitHub and saw your outputs